In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path


In [2]:

INPUT_PATH = Path("processed_data/processed_amazon_support.csv")
OUTPUT_DIR = Path("eda_outputs")


In [3]:
ISSUE_KEYWORDS = {
    "delivery_shipping": [
        "delivery", "delivered", "shipping", "shipment", "arrived",
        "package", "parcel", "carrier", "courier", "tracking", "late"
    ],
    "refund_return": [
        "refund", "return", "replacement", "charge", "charged",
        "money", "payment", "cancel", "canceled", "cancelled"
    ],
    "account_prime": [
        "account", "login", "password", "locked", "prime",
        "membership", "email", "sign in"
    ],
    "device_app": [
        "echo", "kindle", "alexa", "app", "video",
        "fire", "device", "register", "supported"
    ],
    "damaged_missing_item": [
        "damaged", "broken", "missing", "opened", "wrong item",
        "counterfeit", "fake", "condition"
    ],
    "general_support": [
        "help", "support", "customer service", "contact",
        "phone", "chat", "issue", "problem"
    ]
}

In [4]:
def assign_issue_category(text):
    text = str(text).lower()

    category_scores = {}

    for category, keywords in ISSUE_KEYWORDS.items():
        score = 0

        for keyword in keywords:
            if keyword in text:
                score += 1

        category_scores[category] = score

    best_category = max(category_scores, key=category_scores.get)

    if category_scores[best_category] == 0:
        return "other"

    return best_category

In [5]:
def get_top_words(series, top_n=30):
    all_words = " ".join(series.dropna().astype(str)).split()

    simple_stopwords = {
        "the", "a", "an", "and", "or", "to", "for", "of", "in", "on",
        "is", "are", "was", "were", "i", "you", "we", "it", "this",
        "that", "with", "my", "your", "me", "our", "be", "have",
        "has", "do", "can", "not", "but", "so"
    }

    words = [
        word for word in all_words
        if word not in simple_stopwords and len(word) > 2
    ]

    return Counter(words).most_common(top_n)


In [8]:
def save_bar_chart(data, title, xlabel, ylabel, output_path):
    data = list(data)

    if len(data) == 0:
        print(f"No data to plot for: {title}")
        return

    labels = [str(item[0]) for item in data]
    values = [int(item[1]) for item in data]

    plt.figure(figsize=(12, 6))
    plt.bar(labels, values)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

In [ ]:
def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    df = pd.read_csv(INPUT_PATH)

    print("=" * 60)
    print("EDA Report")
    print("=" * 60)

    print(f"Rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nQuestion word count stats:")
    print(df["question_word_count"].describe())

    print("\nAnswer word count stats:")
    print(df["answer_word_count"].describe())

    # Assign issue categories
    df["issue_category"] = df["eda_question"].apply(assign_issue_category)

    category_counts = df["issue_category"].value_counts()

    print("\nIssue category counts:")
    print(category_counts)

    # Save dataset with categories
    output_csv = OUTPUT_DIR / "amazon_support_with_categories.csv"
    df.to_csv(output_csv, index=False)

    # Top words
    top_question_words = get_top_words(df["eda_question"], top_n=30)
    top_answer_words = get_top_words(df["eda_answer"], top_n=30)

    print("\nTop question words:")
    for word, count in top_question_words:
        print(f"{word}: {count}")

    print("\nTop answer words:")
    for word, count in top_answer_words:
        print(f"{word}: {count}")

    # Save charts
    save_bar_chart(
    list(category_counts.items()),
    "Common Customer Support Issue Categories",
    "Issue Category",
    "Number of Questions",
    OUTPUT_DIR / "issue_categories.png"
    )

    save_bar_chart(
        top_question_words,
        "Top Words in Customer Questions",
        "Word",
        "Frequency",
        OUTPUT_DIR / "top_question_words.png"
    )

    save_bar_chart(
        top_answer_words,
        "Top Words in Support Answers",
        "Word",
        "Frequency",
        OUTPUT_DIR / "top_answer_words.png"
    )

    print("\nSaved EDA outputs to:")
    print(OUTPUT_DIR)


if __name__ == "__main__":
    main()

EDA Report
Rows: 123023
Columns: ['question', 'answer', 'company', 'clean_question', 'clean_answer', 'eda_question', 'eda_answer', 'question_word_count', 'answer_word_count']

Missing values:
question               0
answer                 0
company                0
clean_question         0
clean_answer           0
eda_question           0
eda_answer             0
question_word_count    0
answer_word_count      0
dtype: int64

Question word count stats:
count    123023.000000
mean         20.607699
std          10.122036
min           3.000000
25%          14.000000
50%          20.000000
75%          25.000000
max          64.000000
Name: question_word_count, dtype: float64

Answer word count stats:
count    123023.000000
mean         19.713639
std           7.629456
min           3.000000
25%          15.000000
50%          19.000000
75%          22.000000
max          55.000000
Name: answer_word_count, dtype: float64

Issue category counts:
issue_category
delivery_shipping       351

: 